# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Wanoleo/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [6]:
from google.colab import files
uploaded = files.upload()  # select capstone_features.csv

import pandas as pd, numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import roc_auc_score, classification_report

df = pd.read_csv('capstone_features.csv')
feats = ['imp_prev30','visible_queries','rare_share','anon_share','top_query_share','pos_volatility_60d']
model_data = df.dropna(subset=feats)
X, y = model_data[feats], model_data['is_declining']
print(f'{len(model_data):,} rows ready')

Saving capstone_features.csv to capstone_features (1).csv
102,202 rows ready


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [7]:
print("Method: RandomForestClassifier (200 trees).")
print("Why: combines multiple moderate signals non-linearly without heavy tuning,")
print("and handles the mix of skewed/bounded features here reasonably well.")

Method: RandomForestClassifier (200 trees).
Why: combines multiple moderate signals non-linearly without heavy tuning,
and handles the mix of skewed/bounded features here reasonably well.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [8]:
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=model_data['client_hash_id']))
X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
print("Split: GroupShuffleSplit by client_hash_id (75/25).")
print("Why: tests generalization to clients never seen in training —")
print("the real-world condition a production tool would face.")
print(f"Train rows: {len(X_tr):,} | Test rows: {len(X_te):,}")

Split: GroupShuffleSplit by client_hash_id (75/25).
Why: tests generalization to clients never seen in training —
the real-world condition a production tool would face.
Train rows: 50,501 | Test rows: 51,701


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [9]:
model = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1).fit(X_tr, y_tr)
auc = roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])
baseline_auc = roc_auc_score(y_te, model_data.loc[X_te.index, 'pos_volatility_60d'])
print(f'Baseline (volatility alone) AUC: {baseline_auc:.3f}')
print(f'Model (6 features) AUC:          {auc:.3f}')

Baseline (volatility alone) AUC: 0.623
Model (6 features) AUC:          0.705


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [10]:
base_rate = max(y_te.mean(), 1 - y_te.mean())
print(f'Base rate: {base_rate:.3f}')
print(classification_report(y_te, model.predict(X_te), digits=3))
print()
print("Model accuracy sits below the base rate here — expected, since ~68%")
print("of pages are already 'declining'. AUC is the honest metric; it shows")
print("real separation the accuracy number hides.")

Base rate: 0.677
              precision    recall  f1-score   support

           0      0.466     0.668     0.549     16706
           1      0.800     0.635     0.708     34995

    accuracy                          0.646     51701
   macro avg      0.633     0.651     0.629     51701
weighted avg      0.692     0.646     0.657     51701


Model accuracy sits below the base rate here — expected, since ~68%
of pages are already 'declining'. AUC is the honest metric; it shows
real separation the accuracy number hides.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.